# Example of Using BERT MODEL 

In [1]:
from transformers import BertTokenizer, BertModel
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-cased')
model = BertModel.from_pretrained("bert-base-cased")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
def get_BERT_embeddings(text, tokenizer, model):
    encoded_input = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        output = model(**encoded_input)
    return output.last_hidden_state.numpy()

In [3]:
text = "Replace me by any text you'd like."
get_BERT_embeddings(text, tokenizer, model)

array([[[ 0.60232   ,  0.10919464,  0.1417215 , ..., -0.41773635,
          0.6058591 ,  0.17640212],
        [ 0.51185334, -0.47698203,  0.55075073, ..., -0.28141245,
          0.37927824,  0.11556844],
        [ 0.09948041,  0.08669529,  0.08693296, ...,  0.47888252,
         -0.32364357,  0.3121996 ],
        ...,
        [ 0.8080858 , -0.73800963,  0.20007104, ...,  0.7404605 ,
         -0.79981405,  0.6448887 ],
        [ 0.33053648, -0.1957809 ,  0.31480154, ..., -0.05245691,
          0.5358101 ,  0.19870356],
        [ 0.56553894, -0.21757358, -0.47202122, ..., -0.35540923,
          0.61410856, -0.24756138]]], shape=(1, 13, 768), dtype=float32)

In [4]:
tokenizer("I love you.[SEP]You are the best.")

{'input_ids': [101, 146, 1567, 1128, 119, 102, 1192, 1132, 1103, 1436, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [5]:
tokenizer.decode([101, 146, 1567, 1128, 119])

'[CLS] I love you.'

# Load Squad dataset

In [6]:
import json 
with open('data/SQuAD/train-v2.0.json') as f: 
    data_train = json.load(f)
with open('data/SQuAD/dev-v2.0.json') as f:
    data_test = json.load(f)

In [239]:
data_train['data'][0]['paragraphs'][0]

{'qas': [{'question': 'When did Beyonce start becoming popular?',
   'id': '56be85543aeaaa14008c9063',
   'answers': [{'text': 'in the late 1990s', 'answer_start': 269}],
   'is_impossible': False},
  {'question': 'What areas did Beyonce compete in when she was growing up?',
   'id': '56be85543aeaaa14008c9065',
   'answers': [{'text': 'singing and dancing', 'answer_start': 207}],
   'is_impossible': False},
  {'question': "When did Beyonce leave Destiny's Child and become a solo singer?",
   'id': '56be85543aeaaa14008c9066',
   'answers': [{'text': '2003', 'answer_start': 526}],
   'is_impossible': False},
  {'question': 'In what city and state did Beyonce  grow up? ',
   'id': '56bf6b0f3aeaaa14008c9601',
   'answers': [{'text': 'Houston, Texas', 'answer_start': 166}],
   'is_impossible': False},
  {'question': 'In which decade did Beyonce become famous?',
   'id': '56bf6b0f3aeaaa14008c9602',
   'answers': [{'text': 'late 1990s', 'answer_start': 276}],
   'is_impossible': False},
  {'q

In [7]:
data_train['data'][0]['paragraphs'][0]['context'][269:]

'in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".'

In [8]:
data_train['data'][0]['paragraphs'][0]

{'qas': [{'question': 'When did Beyonce start becoming popular?',
   'id': '56be85543aeaaa14008c9063',
   'answers': [{'text': 'in the late 1990s', 'answer_start': 269}],
   'is_impossible': False},
  {'question': 'What areas did Beyonce compete in when she was growing up?',
   'id': '56be85543aeaaa14008c9065',
   'answers': [{'text': 'singing and dancing', 'answer_start': 207}],
   'is_impossible': False},
  {'question': "When did Beyonce leave Destiny's Child and become a solo singer?",
   'id': '56be85543aeaaa14008c9066',
   'answers': [{'text': '2003', 'answer_start': 526}],
   'is_impossible': False},
  {'question': 'In what city and state did Beyonce  grow up? ',
   'id': '56bf6b0f3aeaaa14008c9601',
   'answers': [{'text': 'Houston, Texas', 'answer_start': 166}],
   'is_impossible': False},
  {'question': 'In which decade did Beyonce become famous?',
   'id': '56bf6b0f3aeaaa14008c9602',
   'answers': [{'text': 'late 1990s', 'answer_start': 276}],
   'is_impossible': False},
  {'q

In [352]:
def preprocess_data(data, maxlen=512):
    X_train_text = []
    X_train_token_ids = []
    y_start = []
    y_end = []
    answers_text = []
    not_match = []
    exceed_maxlen = []
    is_impossible = []
    
    for x in data['data']:
        for para in x['paragraphs']:
            for l in para['qas']:
                text = l['question'] + ' [SEP] ' + para['context']
                X_train_text.append(text)
                text_token_ids = tokenizer(text,  padding='max_length', truncation = True)['input_ids']
                n_tokens_question = len(tokenizer(l['question'])['input_ids'])-1 # excluding [SEP]
                X_train_token_ids.append(text_token_ids)
                                        
                if l['is_impossible']:
                    answers_text.append(None)
                    y_start.append(n_tokens_question)
                    y_end.append(n_tokens_question)
                    not_match.append(None)
                    exceed_maxlen.append(None)
                    is_impossible.append(True)
                else:
                    is_impossible.append(False)
                    answers_text.append(l['answers'][0]['text'])
                    answer_token_ids = tokenizer(l['answers'][0]['text'])['input_ids'][1:-1]  # excluding [CLS] and [SEP]
                    n_tokens_answer = len(answer_token_ids) 
                    idx = l['answers'][0]['answer_start']
                    pos_start_answer = len(tokenizer(para['context'][:idx])['input_ids'])-1 # excluding [SEP]
                    if text_token_ids[n_tokens_question+pos_start_answer: n_tokens_question+pos_start_answer+n_tokens_answer] != answer_token_ids:
                        # print(para['context'][:idx], para['context'][idx:])
                        # print("question:",   l['question'])
                        # print("original answer:", l['answers'][0]['text'])
                        # print("segment:", tokenizer.decode(text_token_ids[n_tokens_question+pos_start_answer: n_tokens_question+pos_start_answer+n_tokens_answer]))
                        not_match.append(True)
                    else:
                        not_match.append(False)
                    if n_tokens_question+pos_start_answer+n_tokens_answer < maxlen:
                        y_start.append(n_tokens_question+pos_start_answer)
                        y_end.append(n_tokens_question+pos_start_answer+n_tokens_answer)
                        exceed_maxlen.append(False)
                    else: # if exceed maxlen, not found
                        y_start.append(n_tokens_question)
                        y_end.append(n_tokens_question)
                        exceed_maxlen.append(True)
    return {
        'X_train_text': X_train_text,
        'X_train_token_ids': X_train_token_ids,
        'y_start': y_start,
        'y_end': y_end,
        'answers_text': answers_text,
        'not_match': not_match,
        'exceed_maxlen': exceed_maxlen,
        'is_impossible': is_impossible,
    }

In [353]:
res_train = preprocess_data(data_train)
res_test = preprocess_data(data_test)

Token indices sequence length is longer than the specified maximum sequence length for this model (525 > 512). Running this sequence through the model will result in indexing errors


In [271]:
np.array(res_train['exceed_maxlen']).astype(bool).mean()

np.float64(9.975521604677752e-05)

In [272]:
print(np.array(res_train['not_match']).astype(bool).mean(), np.array(res_train['exceed_maxlen']).astype(bool).mean(), np.array(res_train['is_impossible']).mean())

0.0017495530198973288 9.975521604677752e-05 0.3337809528925176


In [273]:
print(np.array(res_test['not_match']).astype(bool).mean(), np.array(res_test['exceed_maxlen']).astype(bool).mean(), np.array(res_test['is_impossible']).mean())

0.001094921249894719 0.000589572980712541 0.5007159100480081


In [274]:
X_train = torch.tensor(res_train['X_train_token_ids'])
y_s = nn.functional.one_hot(torch.tensor(res_train['y_start']), num_classes=512)
y_e = nn.functional.one_hot(torch.tensor(res_train['y_end']), num_classes=512)

In [276]:
X_train.shape

torch.Size([130319, 512])

In [279]:
model(X_train[:4]).last_hidden_state.shape

torch.Size([4, 512, 768])

In [300]:
from transformers import BertTokenizer, BertModel
import torch
from torch import nn

tokenizer = BertTokenizer.from_pretrained('bert-base-cased')
model = BertModel.from_pretrained("bert-base-cased")

class BertSQuADFT(nn.Module):
    def __init__(self):
        super().__init__()
        self.BERT_model = BertModel.from_pretrained("bert-base-cased")
        self.d_h = self.BERT_model.embeddings.word_embeddings.embedding_dim

        self.start_of_span = nn.Linear(self.d_h, 1)
        self.end_of_span = nn.Linear(self.d_h, 1)
        
    def forward(self, x):
        output = self.BERT_model(x)
        # mask = torch.cumsum(torch.where(x['input_ids']==102,1,0), dim=-1) % 2
        
        logits_start = self.start_of_span(output.last_hidden_state).squeeze(-1)
        logits_end = self.end_of_span(output.last_hidden_state).squeeze(-1)
        return logits_start, logits_end # , mask

    def loss(self, x, y_s, y_e):
        logits_start, logits_end = self.forward(x)
        return nn.functional.cross_entropy(logits_start, y_s) + nn.functional.cross_entropy(logits_end, y_e)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [305]:
modelQA = BertSQuADFT()

modelQA = modelQA.to(device)
X_train = X_train.to(device)
y_s = y_s.to(device).float()
y_e = y_e.to(device).float()

a,b = modelQA(X_train[:4])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [306]:
n_epoch = 3
opt = torch.optim.Adam(params=modelQA.parameters(), lr=5e-5)
batch_size = 16
n_batches = len(X_train)//batch_size + 1


In [309]:
y_s.sum(dim=-1)

tensor([1., 1., 1.,  ..., 1., 1., 1.], device='cuda:0')

In [310]:
for i in range(n_epoch):
    print(f"--------------Epoch {i} --------------------")
    for j in range(n_batches):
        X_batch = X_train[j*batch_size: (j+1)*batch_size]
        ys_batch = y_s[j*batch_size: (j+1)*batch_size]
        ye_batch = y_e[j*batch_size: (j+1)*batch_size]
        
        opt.zero_grad()
        loss = modelQA.loss(X_batch, ys_batch, ye_batch)
        loss.backward()
        opt.step()
        if j%100 == 0:
            print(f'Iteration {j} loss = {loss}')
            

--------------Epoch 0 --------------------
Iteration 0 loss = 12.833660125732422
Iteration 100 loss = 5.083580017089844
Iteration 200 loss = 5.8834309577941895
Iteration 300 loss = 3.440308094024658
Iteration 400 loss = 4.056471347808838
Iteration 500 loss = 3.021771192550659
Iteration 600 loss = 4.9300537109375
Iteration 700 loss = 3.6224942207336426
Iteration 800 loss = 2.843116044998169
Iteration 900 loss = 1.6288375854492188
Iteration 1000 loss = 6.523856163024902
Iteration 1100 loss = 4.1068596839904785
Iteration 1200 loss = 6.663636684417725
Iteration 1300 loss = 4.024290084838867
Iteration 1400 loss = 2.461845636367798
Iteration 1500 loss = 3.737046480178833
Iteration 1600 loss = 3.3078742027282715
Iteration 1700 loss = 3.046452522277832
Iteration 1800 loss = 1.6402279138565063
Iteration 1900 loss = 3.9808804988861084
Iteration 2000 loss = 1.9232969284057617
Iteration 2100 loss = 4.036892890930176
Iteration 2200 loss = 2.2822818756103516
Iteration 2300 loss = 2.808694362640381
I

KeyboardInterrupt: 

In [315]:
idx = np.random.choice(range(len(X_train)), size=100)

In [317]:
modelQA.eval()
with torch.no_grad():
    logits_ys, logits_ye = modelQA(X_train[idx])

In [321]:
logits_ys.unsqueeze(-2).shape

torch.Size([100, 1, 512])

In [361]:
logits_ys.argmax(dim=-1)

tensor([ 25,  70,  16,  82,  34,  11,  17, 181, 172,  13,  13,  45, 129, 115,
         63,  51,  15,  15,   7,  72,  15,  21, 155, 235,  13,  13,  25,  10,
         12,  19,  12, 180, 133,  14,  21,  90,  11, 132,  83,  20,  60,  29,
         91,  17, 163,  36,  13,  18, 145, 196,  62,  13,  53, 107,  35,  10,
        122,  63,  33,  14, 155,  16,  26, 170,  58,  78,  19,  13,  93,  16,
         10, 290,  60, 294,  22,  38,  27,  14,  20, 144, 136, 117,  51,  14,
          8,  51,  64,  16,  62,  26,  79, 105,  50, 113,  19,  45,   9,  18,
        150,  59], device='cuda:0')

In [362]:
logits_ye.argmax(dim=-1)

tensor([ 29,  73,  20,  83,  36,  11, 276,   9, 174,  13, 133,  46, 135, 124,
         64,  55, 398, 105, 208,  73,  15,  23, 159,  16,  13, 225,  37,  14,
        134,  19,  12, 182, 138,  14, 233, 103,  11, 135,  84,  21,  61,  33,
         95,  20,  16,  40,  17,  18,  10,  11,  66,  43,  56, 342,  37, 125,
        123,  19,  42,  23,  17,  50,  40,  10,  23,  80, 155,  13,  96,  16,
         16, 296,   9, 298,  10,  42,  34,  14,  20, 148, 138, 123,  52,  14,
        192,  11,  71, 201,  67,  34,  91,  13,  52, 115,  19, 196, 186, 168,
        152,  67], device='cuda:0')

In [367]:
maxidx = torch.triu(logits_ys.unsqueeze(-1) + logits_ye.unsqueeze(-2)).reshape(logits_ye.shape[0],-1).argmax(dim=-1)
maxi, maxj = maxidx//512, maxidx%512

In [370]:
ans_pred = [tokenizer.decode(X_train[idx[i]][maxi[i]:maxj[i]].cpu().numpy()) for i in range(100)]

In [371]:
ans_pred

['August 18, 2011',
 'Naples, Italy',
 'United States Census Bureau',
 'Julian',
 'The military',
 '',
 "[SEP] However, most of the major FBS teams annually schedule early season non - conference preseason home games against lesser opponents that are lower - tier FBS, Football Championship, or Division II schools, which often result in lopsided victories in favor of the FBS teams and act as exhibition games in all but name, though they additionally provide a large appearance fee and at least one guaranteed television appearance for the smaller school. These games also receive the same criticism as NFL exhibition games, but instead it is targeted to schools scheduling low - quality opponents and the simplicity for a team to run up the score against a weak opponent. However, these games are susceptible to backfiring, resulting in damage in poll position and public perception, especially if the higher ranked team loses, although the mere act of scheduling a weak opponent is harmful to a t

In [372]:
ans = [res_train['answers_text'][idx[i]] for i in range(100)]

In [373]:
ans

['August 18, 2011',
 'Italy',
 'United States Census Bureau',
 'Julian',
 'The military',
 'Muslim',
 None,
 None,
 'plant breeding',
 'interpretations',
 None,
 'Cyprus',
 'maternal immunization hypothesis',
 'Arianism, Pelagianism, Donatism, Marcionism and Montanism',
 'two distinct Jewish populations',
 '7,855',
 None,
 'puts the treaty and all of its obligations in action',
 None,
 '1880',
 'home schooling',
 'western',
 'Basic Officer Leaders Course',
 'the American Revolutionary War',
 None,
 None,
 'The end',
 'Crystal Bowersox',
 None,
 'people of color',
 'folk music',
 'Fighting Spirit',
 'buried or scattered over water',
 None,
 None,
 'court applied the Establishment Clause to the laws of a state',
 None,
 'critical bandwidths',
 'Berlin',
 'Lions',
 'Israel',
 '100–110 m',
 'Anna Lelkes',
 None,
 'current flow across the slightly resistive supply lines or the electrolyte',
 'a rabid dog',
 '58.1%',
 None,
 None,
 None,
 None,
 'DC',
 'the Battle of Britain',
 '1794',
 'pri

In [331]:
maxi

tensor([ 29,  73,  20,  83,  36,  11,  17,   9, 174,  13,  13,  46,   8, 124,
         64,  11,  15, 105, 208,  32,  15,  15, 159,  16,  13,  13,  30,   9,
         12,  19,  12, 182,  14,  14, 233, 103,  11,  14,  84,  19,   6,  14,
         95,  20,  16,  40,  12,  18,  10,  11,  27,  43,  56,  17,   7,  10,
        123,  19,  15,  23,  17,  50,  11,  10,  23,  13,  19,  13,  17,  16,
         16,  28,   9, 298,  10,  42,  34,  14,  20, 148,   9, 123,  11,  14,
          8,  11,  71, 201,  67,  34,  91,  13,  52,  16,  19,  22, 186, 168,
         22,  67], device='cuda:0')

In [332]:
maxj

tensor([159, 159,  81, 255, 140,  11,  17,   9, 264,  13,  13, 125, 129, 252,
        133,  51,  15, 105, 208,  72,  15,  21, 195, 235,  13,  13,  31,  10,
         12,  19,  12, 284, 133,  14, 233, 173,  11, 132, 194,  20,  60,  29,
        180, 135,  16, 201,  13,  18,  10, 196,  62, 289,  70, 107,  35,  10,
        163,  63,  33, 256, 155, 136,  26, 170,  58,  78,  19,  13,  93,  16,
        383, 290,  60, 309,  22,  66, 121,  14,  20, 204, 136, 163,  51,  14,
          8,  51, 282, 201, 135, 143,  98, 105, 151, 113,  19,  45, 186, 168,
        150, 171], device='cuda:0')

In [335]:
torch.where(y_s[idx]==1)

(tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
         36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
         54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71,
         72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89,
         90, 91, 92, 93, 94, 95, 96, 97, 98, 99], device='cuda:0'),
 tensor([ 25,  72,  16,  82,  34,  28,  17,   9, 172, 123,  13,  45, 129, 126,
          63,  51,  15,  35,   7,  31,  94,  21, 155,  18,  13,  13,  25,  10,
          12, 175,  23, 180, 133,  14,  21,  91,  11, 132,  83,  20,  60,  29,
          91,  17,  76,  35,  13,  18,  10,  11,  27,  42,  52, 107,  35,  10,
          21,  19,  33,  22,  17,  16,  11, 164,  58,  78,  82,  13,  93, 128,
          11, 289,   9, 294,  22,  34,  27,  82,  20,  14, 136, 117,  51,  46,
           8,  51, 225, 150,  62,  26,  79,  13,  50,

In [358]:
torch.where(y_e[idx]==1)

(tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
         36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
         54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71,
         72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89,
         90, 91, 92, 93, 94, 95, 96, 97, 98, 99], device='cuda:0'),
 tensor([ 29,  73,  20,  83,  36,  29,  17,   9, 174, 124,  13,  46, 135, 147,
          67,  55,  15,  45,   7,  32,  96,  22, 159,  22,  13,  13,  27,  14,
          12, 178,  25, 182, 138,  14,  21, 103,  11, 135,  84,  21,  61,  33,
          95,  17,  90,  40,  17,  18,  10,  11,  27,  43,  56, 108,  37,  10,
          21,  19,  42,  23,  17,  16,  11, 169,  59,  80,  91,  13,  96, 131,
          16, 291,   9, 298,  25,  42,  34,  84,  20,  14, 138, 123,  52,  50,
           8,  55, 226, 177,  67,  34,  91,  13,  52,

In [90]:
sum([k.numel() for k in model.parameters()])


108310272

In [215]:
np.array(not_match).mean()

np.float64(0.0024763594061344607)

In [ ]:
y_start = torch.tensor(y_start)
y_end = torch.tensor(y_end)
y_start[y_start>=512] = 

In [219]:
nn.functional.one_hot(torch.tensor(y_start), num_classes=512)


RuntimeError: Class values must be smaller than num_classes.

In [222]:
(np.array(y_end)>=512).mean()

np.float64(9.975521604677752e-05)

In [217]:
torch.zeros((len(y_start), 512))

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [197]:
text_token_ids[n_tokens_question+pos_start_answer: n_tokens_question+pos_start_answer+n_tokens_answer]

[20278, 1568]

In [198]:
answer_token_ids

[20278, 1568]

In [189]:
answer_token_ids

[20278, 1568]

In [188]:
pos_start_answer

3

In [184]:
X_train_token_ids[n_tokens_question]

102

In [178]:
n_tokens_question

16

In [176]:
l['question']

"What could the sun's energy do to help limit climate change?"

In [101]:
tokenizer(answers_text[:10])

{'input_ids': [[101, 1107, 1103, 1523, 3281, 102], [101, 4241, 1105, 5923, 102], [101, 1581, 102], [101, 4666, 117, 2245, 102], [101, 1523, 3281, 102], [101, 16784, 112, 188, 6405, 102], [101, 20924, 1193, 1107, 2185, 102], [101, 15112, 5773, 25384, 102], [101, 1523, 3281, 102], [101, 1730, 2483, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 1]]}

In [157]:
token_ids = tokenizer(X_train_text[2])['input_ids']
start_pos = tokenizer(X_train
s = ' '.join([str(x) for x in token_ids])
target = ' '.join([str(x) for x in tokenizer(answers_text[2])['input_ids'][1:-1]])

In [161]:
idx = s.find('102')
q_len = len(s[:idx].split(' '))-1
a_len = len(s[idx:].split(' '))
t_len = len(target.split(' '))
pos = s[q_len:].find(target)
n_tokens = len(s[q_len:pos].split(' '))
print(target, tokenizer(X_train_text[2])['input_ids'][q_len+n_tokens+1:q_len+n_tokens+1+t_len])

1581 [2829]


In [172]:
pos

663

In [170]:
s[q_len:pos+10]

'96 1320 2093 1817 16784 112 188 6405 1105 1561 170 3444 2483 136 102 24041 144 22080 25384 118 5007 113 120 100 120 17775 118 162 11414 118 1474 114 113 1255 1347 125 117 2358 114 1110 1126 1237 2483 117 5523 117 1647 2451 1105 3647 119 3526 1105 2120 1107 4666 117 2245 117 1131 1982 1107 1672 4241 1105 5923 6025 1112 170 2027 117 1105 3152 1106 8408 1107 1103 1523 3281 1112 1730 2483 1104 155 111 139 1873 118 1372 16784 112 188 6405 119 2268 15841 1118 1123 1401 117 15112 5773 25384 117 1103 1372 1245 1141 1104 1103 1362 112 188 1436 118 4147 1873 2114 1104 1155 1159 119 2397 14938 1486 1103 1836 1104 24041 112 188 1963 1312 117 20924 1193 1107 21'

In [162]:
n_tokens

136

In [168]:
tokenizer(X_train_text[2])['input_ids'][q_len+n_tokens:]

[117,
 2829,
 1421,
 8645,
 2763,
 1105,
 2081,
 1103,
 4192,
 4126,
 1620,
 1295,
 118,
 1141,
 3896,
 107,
 11722,
 1107,
 2185,
 107,
 1105,
 107,
 6008,
 4596,
 107,
 119,
 102]

In [136]:
a_len

163

In [140]:
tokenizer(X_train_text[2])['input_ids'][:q_len]

[101,
 1332,
 1225,
 24896,
 1320,
 2093,
 1817,
 16784,
 112,
 188,
 6405,
 1105,
 1561,
 170,
 3444,
 2483,
 136]

In [130]:
s[s.find('102'):].find(target)

598

In [123]:
s

'101 1332 1225 24896 1320 2093 1817 16784 112 188 6405 1105 1561 170 3444 2483 136 102 24041 144 22080 25384 118 5007 113 120 100 120 17775 118 162 11414 118 1474 114 113 1255 1347 125 117 2358 114 1110 1126 1237 2483 117 5523 117 1647 2451 1105 3647 119 3526 1105 2120 1107 4666 117 2245 117 1131 1982 1107 1672 4241 1105 5923 6025 1112 170 2027 117 1105 3152 1106 8408 1107 1103 1523 3281 1112 1730 2483 1104 155 111 139 1873 118 1372 16784 112 188 6405 119 2268 15841 1118 1123 1401 117 15112 5773 25384 117 1103 1372 1245 1141 1104 1103 1362 112 188 1436 118 4147 1873 2114 1104 1155 1159 119 2397 14938 1486 1103 1836 1104 24041 112 188 1963 1312 117 20924 1193 1107 2185 113 1581 114 117 1134 1628 1123 1112 170 3444 2360 4529 117 2829 1421 8645 2763 1105 2081 1103 4192 4126 1620 1295 118 1141 3896 107 11722 1107 2185 107 1105 107 6008 4596 107 119 102'

In [124]:
answers_text[2]

'2003'

In [125]:
target

'1581'

In [98]:
X_train_text[5][start_idx[5]:]

' the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".'

In [57]:
import numpy as np
np.where(np.array(is_impossible)==True)

(array([  2075,   2076,   2077, ..., 130316, 130317, 130318],
       shape=(43498,)),)

In [59]:
43498.0/len(is_impossible)

0.3337809528925176

In [62]:
len(X_train_text)

130319

In [65]:
len(is_impossible)

130319

In [58]:
answers_text[2075]

In [29]:
X_train_encoded = tokenizer(X_train_text[:16], padding='max_length', truncation = True, return_tensors='pt')


In [44]:
X_train_encoded

{'input_ids': tensor([[ 101, 1332, 1225,  ...,    0,    0,    0],
        [ 101, 1327, 1877,  ...,    0,    0,    0],
        [ 101, 1332, 1225,  ...,    0,    0,    0],
        ...,
        [ 101, 1327, 1108,  ...,    0,    0,    0],
        [ 101, 1327, 1108,  ...,    0,    0,    0],
        [ 101, 1258, 1123,  ...,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [37]:
X_batched = {'input_ids': X_train_encoded['input_ids'][:,:16], 
             'token_type_ids': X_train_encoded['token_type_ids'][:,:16], 
             'attention_mask': X_train_encoded['attention_mask'][:,:16], 
            }

In [38]:
device = torch.device('cuda')

In [39]:
model = model.to(device)

In [40]:
X_batched_device = {k: v.to(device) for k, v in X_batched.items()}

In [41]:
torch.cuda.empty_cache()

In [42]:
out = model(**X_batched_device)

In [43]:
out

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.1177,  0.1175,  0.0213,  ...,  0.2041,  0.2689,  0.1279],
         [-0.2791, -0.0660,  0.5491,  ...,  0.0455, -0.0456,  0.0842],
         [-0.2304,  0.0381,  0.0380,  ...,  1.0772,  0.0232,  0.5456],
         ...,
         [-0.0846,  0.3906, -0.2636,  ...,  0.0735, -0.9815, -0.3947],
         [-0.0452,  0.0433, -0.1566,  ...,  0.2041,  0.6949,  0.5109],
         [-0.4845, -0.3800, -0.5209,  ...,  0.9995,  0.5556,  0.1666]],

        [[ 0.4394,  0.1369, -0.1498,  ..., -0.1345,  0.0490, -0.0652],
         [ 0.3357, -0.5761,  0.5472,  ..., -0.2445, -0.1607, -0.4017],
         [ 0.7767,  0.1826, -0.0838,  ...,  0.0089, -0.3067, -0.0133],
         ...,
         [ 0.5634, -0.1426,  0.4714,  ...,  0.4766,  0.5279,  0.1001],
         [ 0.2976, -0.3486, -0.1525,  ...,  0.2664, -0.0915,  0.0692],
         [ 0.5890, -0.2464,  0.0527,  ...,  0.3973,  0.3987, -0.0987]],

        [[ 0.1868,  0.0745, -0.3373,  ...,  0.3115,  

In [109]:
(X_train_encoded['input_ids'][:,-1]==0).float().mean()

tensor(0.9982)

In [ ]:
y_s_train = []

In [8]:
d['data'][0]

{'title': 'Beyoncé',
 'paragraphs': [{'qas': [{'question': 'When did Beyonce start becoming popular?',
     'id': '56be85543aeaaa14008c9063',
     'answers': [{'text': 'in the late 1990s', 'answer_start': 269}],
     'is_impossible': False},
    {'question': 'What areas did Beyonce compete in when she was growing up?',
     'id': '56be85543aeaaa14008c9065',
     'answers': [{'text': 'singing and dancing', 'answer_start': 207}],
     'is_impossible': False},
    {'question': "When did Beyonce leave Destiny's Child and become a solo singer?",
     'id': '56be85543aeaaa14008c9066',
     'answers': [{'text': '2003', 'answer_start': 526}],
     'is_impossible': False},
    {'question': 'In what city and state did Beyonce  grow up? ',
     'id': '56bf6b0f3aeaaa14008c9601',
     'answers': [{'text': 'Houston, Texas', 'answer_start': 166}],
     'is_impossible': False},
    {'question': 'In which decade did Beyonce become famous?',
     'id': '56bf6b0f3aeaaa14008c9602',
     'answers': [{'text

In [6]:
d.keys()

dict_keys(['version', 'data'])